# ETL do conjunto de dados do Banco de Preços em Saúde

Análise estrutural dos arquivos anuais do BPS (2020 a 2026), realizada antes da concatenação.

In [1]:
import glob
from pathlib import Path

import pandas as pd

identifier_columns = [
    'codigo_br',
    'anvisa',
    'cnpj_instituicao',
    'cnpj_fornecedor',
    'cnpj_fabricante',
]

# Funciona quando o Jupyter é iniciado na raiz ou na pasta notebooks.
project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent

data_dir = project_dir / 'data'
path_files = sorted(glob.glob(str(data_dir / '*.csv')))

if not path_files:
    raise FileNotFoundError(f'Nenhum arquivo CSV encontrado em: {data_dir}')

print(f'{len(path_files)} arquivos encontrados:')
for file_path in path_files:
    print(f'- {Path(file_path).name}')

7 arquivos encontrados:
- 2020.csv
- 2021.csv
- 2022.csv
- 2023.csv
- 2024.csv
- 2025.csv
- 2026.csv


## 1. Análise estrutural antes da concatenação

Cada arquivo é carregado separadamente. O resumo registra dimensões, nulos e duplicatas, enquanto as colunas e os tipos inferidos são preservados para comparação entre os anos.

In [2]:
dataframes = {}
columns_by_file = {}
dtypes_by_file = {}
structural_records = []

for file_path in path_files:
    path = Path(file_path)
    df = pd.read_csv(
        path,
        sep=';',
        encoding='utf-8',
        low_memory=False,
        dtype={column: 'string' for column in identifier_columns},
    )

    null_count = int(df.isna().sum().sum())
    total_cells = df.shape[0] * df.shape[1]

    structural_records.append({
        'arquivo': path.name,
        'ano': path.stem,
        'linhas': df.shape[0],
        'colunas': df.shape[1],
        'valores_nulos': null_count,
        'percentual_nulos': null_count / total_cells if total_cells else 0,
        'colunas_com_nulos': int(df.isna().any().sum()),
        'linhas_duplicadas': int(df.duplicated().sum())
    })

    columns_by_file[path.name] = df.columns.tolist()
    dtypes_by_file[path.name] = df.dtypes.astype(str).to_dict()
    df['arquivo_origem'] = path.name
    dataframes[path.name] = df

structural_summary = pd.DataFrame(structural_records)

structural_summary['percentual_nulos'] = (
    structural_summary['percentual_nulos'] * 100
).round(2)

display(structural_summary)

,arquivo,ano,linhas,colunas,valores_nulos,percentual_nulos,colunas_com_nulos,linhas_duplicadas
0,2020.csv,2020,84819,25,202254,9.54,6,0
1,2021.csv,2021,83622,25,197216,9.43,7,0
2,2022.csv,2022,88991,25,213444,9.59,7,0
3,2023.csv,2023,31992,25,69545,8.70,6,0
4,2024.csv,2024,26258,25,52139,7.94,8,16
5,2025.csv,2025,26215,25,46577,7.11,7,1
6,2026.csv,2026,819,25,1304,6.37,4,2


In [3]:
reference_file = Path(path_files[0]).name
reference_columns = columns_by_file[reference_file]
reference_dtypes = dtypes_by_file[reference_file]
schema_records = []

for file_name in dataframes:
    current_columns = columns_by_file[file_name]
    current_dtypes = dtypes_by_file[file_name]
    missing_columns = sorted(set(reference_columns) - set(current_columns))
    extra_columns = sorted(set(current_columns) - set(reference_columns))
    common_columns = set(reference_columns) & set(current_columns)
    divergent_types = sorted(
        column
        for column in common_columns
        if current_dtypes[column] != reference_dtypes[column]
    )

    schema_records.append({
        'arquivo': file_name,
        'mesmas_colunas': not missing_columns and not extra_columns,
        'mesma_ordem': current_columns == reference_columns,
        'colunas_ausentes': ', '.join(missing_columns) or '-',
        'colunas_extras': ', '.join(extra_columns) or '-',
        'tipos_divergentes': ', '.join(divergent_types) or '-',
    })

schema_comparison = pd.DataFrame(schema_records)
type_matrix = pd.DataFrame(dtypes_by_file).sort_index()

print(f'Arquivo de referência para a comparação: {reference_file}')
display(schema_comparison)
display(type_matrix)

if schema_comparison['mesmas_colunas'].all():
    print('Estruturas compatíveis: todos os arquivos possuem o mesmo conjunto de colunas.')
else:
    print('Atenção: existem colunas ausentes ou adicionais. Revise antes de concatenar.')

Arquivo de referência para a comparação: 2020.csv


,arquivo,mesmas_colunas,mesma_ordem,colunas_ausentes,colunas_extras,tipos_divergentes
0,2020.csv,True,True,-,-,-
1,2021.csv,True,True,-,-,-
2,2022.csv,True,True,-,-,-
3,2023.csv,True,True,-,-,-
4,2024.csv,True,True,-,-,-
5,2025.csv,True,True,-,-,-
6,2026.csv,True,True,-,-,-


,2020.csv,2021.csv,2022.csv,2023.csv,2024.csv,2025.csv,2026.csv
ano_compra,int64,int64,int64,int64,int64,int64,int64
anvisa,string,string,string,string,string,string,string
capacidade,float64,float64,float64,float64,float64,float64,float64
cnpj_fabricante,string,string,string,string,string,string,string
cnpj_fornecedor,string,string,string,string,string,string,string
cnpj_instituicao,string,string,string,string,string,string,string
codigo_br,string,string,string,string,string,string,string
compra,str,str,str,str,str,str,str
descricao_catmat,str,str,str,str,str,str,str
esfera,str,str,str,str,str,str,str


Estruturas compatíveis: todos os arquivos possuem o mesmo conjunto de colunas.


## 2. Concatenação das bases

A concatenação só é executada quando todos os arquivos apresentam o mesmo conjunto de colunas. Diferenças de ordem não causam perda, pois o pandas alinha os dados pelo nome das colunas.

In [4]:
if not schema_comparison['mesmas_colunas'].all():
    raise ValueError('Concatenação interrompida: foram encontradas estruturas incompatíveis.')

bps_raw = pd.concat(dataframes.values(), ignore_index=True, sort=False)

print(f'{len(dataframes)} arquivos concatenados.')
print(f'{len(bps_raw):,} registros e {bps_raw.shape[1]} colunas na base bruta consolidada.')
display(bps_raw.head())

7 arquivos concatenados.
342,716 registros e 26 colunas na base bruta consolidada.


,ano_compra,nome_instituicao,esfera,cnpj_instituicao,municipio_instituicao,uf,compra,insercao,codigo_br,descricao_catmat,...,unidade_medida,unidade_fornecimento_capacidade,cnpj_fornecedor,fornecedor,cnpj_fabricante,fabricante,qtd_itens_comprados,preco_unitario,preco_total,arquivo_origem
0,2020,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18.478.187/0001-07,MARABA,PA,01/01/2020,19/01/2024,270019,"GLICONATO DE CÁLCIO, DOSAGEM:10%, APRESENTAÇÃO...",...,ML,AMPOLA 10.00 ML,04.949.905/0001-63,F CARDOSO E CIA LTDA,01.571.702/0001-98,HALEX ISTAR INDUSTRIA FARMACEUTICA SA,9750,4.500,43875.0,2020.csv
1,2020,FUNDO MUNICIPAL DE SAUDE - MUNICIPIO DE ALTO P...,MUNICIPAL,08.533.932/0001-01,ALTO PARANA,PR,01/01/2020,13/03/2020,243488,"BOLSA VENTILAÇÃO PULMONAR, MATERIAL:BORRACHA, ...",...,NaN,UNIDADE,06.974.929/0001-06,NOROESTE MEDICAMENTOS LTDA,73.856.593/0001-66,"PRATI, DONADUZZI & CIA LTDA",3,6.900,20.7,2020.csv
2,2020,MUNICIPIO DE AMERICO BRASILIENSE,MUNICIPAL,43.976.166/0001-50,AMERICO BRASILIENSE,SP,01/01/2020,06/10/2020,267567,"CARVEDILOL, DOSAGEM:25 MG",...,NaN,COMPRIMIDO,01.140.868/0001-50,CIRURGICA OLIMPIO LTDA,61.150.447/0001-31,LABORATORIOS BALDACCI LTDA EM RECUPERACAO JUDI...,36000,0.155,5580.0,2020.csv
3,2020,MUNICIPIO DE AMERICO BRASILIENSE,MUNICIPAL,43.976.166/0001-50,AMERICO BRASILIENSE,SP,01/01/2020,06/10/2020,271356,"ALPRAZOLAM, DOSAGEM:1 MG",...,NaN,COMPRIMIDO,01.140.868/0001-50,CIRURGICA OLIMPIO LTDA,05.254.971/0001-81,ZYDUS NIKKHO FARMACEUTICA LTDA,240000,0.099,23760.0,2020.csv
4,2020,MUNICIPIO DE AMERICO BRASILIENSE,MUNICIPAL,43.976.166/0001-50,AMERICO BRASILIENSE,SP,01/01/2020,06/10/2020,267565,"CARVEDILOL, DOSAGEM:6,25 MG",...,NaN,COMPRIMIDO,01.140.868/0001-50,CIRURGICA OLIMPIO LTDA,61.150.447/0001-31,LABORATORIOS BALDACCI LTDA EM RECUPERACAO JUDI...,36000,0.086,3096.0,2020.csv


## 3. Diagnóstico de qualidade

O dicionário oficial define `compra` e `insercao` como datas; `codigo_br`, `anvisa` e CNPJs como identificadores; `capacidade`, `qtd_itens_comprados`, `preco_unitario` e `preco_total` como medidas numéricas. A base também possui `esfera` e `unidade_fornecimento_capacidade`, que não estão descritas de forma inequívoca no documento de metadados.

A base bruta é preservada em `bps_raw`. As transformações serão aplicadas somente a uma cópia chamada `bps_clean`.

In [5]:
date_columns = ['compra', 'insercao']
numeric_columns = [
    'capacidade',
    'qtd_itens_comprados',
    'preco_unitario',
    'preco_total',
]
cnpj_columns = [
    'cnpj_instituicao',
    'cnpj_fornecedor',
    'cnpj_fabricante',
]

null_report_raw = (
    pd.DataFrame({
        'valores_nulos': bps_raw.isna().sum(),
        'percentual_nulos': bps_raw.isna().mean().mul(100).round(2),
    })
    .query('valores_nulos > 0')
    .sort_values('percentual_nulos', ascending=False)
)

diagnostic_records = []

for column in date_columns:
    parsed = pd.to_datetime(bps_raw[column], format='%d/%m/%Y', errors='coerce')
    diagnostic_records.append({
        'verificacao': f'{column}: datas não convertíveis',
        'registros': int((bps_raw[column].notna() & parsed.isna()).sum()),
    })

for column in numeric_columns:
    converted = pd.to_numeric(bps_raw[column], errors='coerce')
    diagnostic_records.extend([
        {
            'verificacao': f'{column}: valores não numéricos',
            'registros': int((bps_raw[column].notna() & converted.isna()).sum()),
        },
        {
            'verificacao': f'{column}: valores negativos',
            'registros': int(converted.lt(0).sum()),
        },
    ])

for column in cnpj_columns:
    digits = bps_raw[column].str.replace(r'\D', '', regex=True)
    diagnostic_records.append({
        'verificacao': f'{column}: tamanho diferente de 14 dígitos',
        'registros': int((digits.notna() & digits.str.len().ne(14)).sum()),
    })

raw_duplicates = int(bps_raw.duplicated().sum())
diagnostic_records.append({
    'verificacao': 'linhas exatamente duplicadas',
    'registros': raw_duplicates,
})

diagnostic_report = pd.DataFrame(diagnostic_records)
display(null_report_raw)
display(diagnostic_report)

,valores_nulos,percentual_nulos
capacidade,218035,63.62
unidade_medida,218035,63.62
generico,172037,50.20
anvisa,172037,50.20
insercao,2128,0.62
nome_instituicao,157,0.05
unidade_fornecimento,25,0.01
unidade_fornecimento_capacidade,25,0.01


,verificacao,registros
0,compra: datas não convertíveis,0
1,insercao: datas não convertíveis,0
2,capacidade: valores não numéricos,0
3,capacidade: valores negativos,0
4,qtd_itens_comprados: valores não numéricos,0
5,qtd_itens_comprados: valores negativos,0
6,preco_unitario: valores não numéricos,0
7,preco_unitario: valores negativos,0
8,preco_total: valores não numéricos,0
9,preco_total: valores negativos,0


## 4. Limpeza auditável

As regras abaixo preservam os dados originais, não imputam medidas ausentes e não descartam anomalias que exigem investigação. Duplicatas exatas são removidas da cópia limpa; datas suspeitas e divergências de valor recebem indicadores de qualidade.

In [7]:
bps_clean = bps_raw.copy()
treatment_records = []

# Normaliza espaços em campos textuais sem alterar maiúsculas, acentos ou conteúdo.
text_columns = bps_clean.select_dtypes(include=['object', 'string']).columns
for column in text_columns:
    original = bps_clean[column].astype('string')
    normalized = original.str.replace(r'\s+', ' ', regex=True).str.strip()
    changed = int((original.notna() & original.ne(normalized)).sum())
    bps_clean[column] = normalized
    if changed:
        treatment_records.append({
            'tratamento': 'normalização de espaços',
            'coluna': column,
            'registros_afetados': changed,
        })

# Mantém CNPJs como texto de 14 dígitos, preservando zeros à esquerda.
for column in cnpj_columns:
    normalized = bps_clean[column].str.replace(r'\D', '', regex=True)
    invalid_mask = normalized.notna() & normalized.str.len().ne(14)
    changed = int(bps_clean[column].notna().sum())
    bps_clean[column] = normalized.mask(invalid_mask)
    treatment_records.append({
        'tratamento': 'CNPJ convertido para 14 dígitos',
        'coluna': column,
        'registros_afetados': changed,
    })

# Converte datas e medidas conforme o dicionário oficial.
for column in date_columns:
    bps_clean[column] = pd.to_datetime(
        bps_clean[column],
        format='%d/%m/%Y',
        errors='coerce',
    )

for column in numeric_columns:
    bps_clean[column] = pd.to_numeric(bps_clean[column], errors='coerce')

bps_clean['ano_compra'] = pd.to_numeric(
    bps_clean['ano_compra'], errors='coerce'
).astype('Int64')

# Preenche apenas rótulos categóricos úteis ao dashboard; medidas ficam nulas.
categorical_fill = {
    'nome_instituicao': 'INSTITUIÇÃO NÃO INFORMADA',
    'generico': 'NÃO INFORMADO',
    'unidade_fornecimento': 'NÃO INFORMADO',
    'unidade_fornecimento_capacidade': 'NÃO INFORMADO',
}
for column, replacement in categorical_fill.items():
    affected = int(bps_clean[column].isna().sum())
    bps_clean[column] = bps_clean[column].fillna(replacement)
    treatment_records.append({
        'tratamento': f'nulo substituído por {replacement}',
        'coluna': column,
        'registros_afetados': affected,
    })

# Remove somente duplicatas exatas, mantendo a primeira ocorrência.
duplicates_after_normalization = int(bps_clean.duplicated().sum())
bps_clean = bps_clean.drop_duplicates().reset_index(drop=True)
treatment_records.append({
    'tratamento': 'remoção de duplicatas exatas',
    'coluna': 'todas',
    'registros_afetados': duplicates_after_normalization,
})

# Cria indicadores para anomalias que devem ser investigadas, não apagadas.
bps_clean['flag_insercao_anterior_compra'] = (
    bps_clean['insercao'].notna()
    & bps_clean['compra'].notna()
    & bps_clean['insercao'].lt(bps_clean['compra'])
)

calculated_total = bps_clean['qtd_itens_comprados'] * bps_clean['preco_unitario']
price_tolerance = 0.01 + bps_clean['preco_total'].abs() * 1e-6
bps_clean['flag_preco_total_divergente'] = (
    bps_clean[['qtd_itens_comprados', 'preco_unitario', 'preco_total']]
    .notna()
    .all(axis=1)
    & bps_clean['preco_total'].sub(calculated_total).abs().gt(price_tolerance)
)

treatment_log = pd.DataFrame(treatment_records)
display(treatment_log)

,tratamento,coluna,registros_afetados
0,normalização de espaços,nome_instituicao,1974
1,normalização de espaços,descricao_catmat,483
2,normalização de espaços,fornecedor,1960
3,normalização de espaços,fabricante,79
4,CNPJ convertido para 14 dígitos,cnpj_instituicao,342716
5,CNPJ convertido para 14 dígitos,cnpj_fornecedor,342716
6,CNPJ convertido para 14 dígitos,cnpj_fabricante,342716
7,nulo substituído por INSTITUIÇÃO NÃO INFORMADA,nome_instituicao,157
8,nulo substituído por NÃO INFORMADO,generico,172037
9,nulo substituído por NÃO INFORMADO,unidade_fornecimento,25


## 5. Validação da base limpa

In [8]:
validation_report = pd.DataFrame([
    {
        'verificacao': 'registros na base bruta',
        'resultado': len(bps_raw),
    },
    {
        'verificacao': 'duplicatas removidas',
        'resultado': duplicates_after_normalization,
    },
    {
        'verificacao': 'registros na base limpa',
        'resultado': len(bps_clean),
    },
    {
        'verificacao': 'datas da compra inválidas',
        'resultado': int(bps_clean['compra'].isna().sum()),
    },
    {
        'verificacao': 'ano divergente da data da compra',
        'resultado': int((
            bps_clean['compra'].notna()
            & bps_clean['ano_compra'].notna()
            & bps_clean['compra'].dt.year.ne(bps_clean['ano_compra'])
        ).sum()),
    },
    {
        'verificacao': 'inserções anteriores à compra',
        'resultado': int(bps_clean['flag_insercao_anterior_compra'].sum()),
    },
    {
        'verificacao': 'preços totais divergentes da fórmula',
        'resultado': int(bps_clean['flag_preco_total_divergente'].sum()),
    },
    {
        'verificacao': 'quantidades ou preços negativos',
        'resultado': int(
            bps_clean[['qtd_itens_comprados', 'preco_unitario', 'preco_total']]
            .lt(0)
            .any(axis=1)
            .sum()
        ),
    },
])

assert len(bps_clean) == len(bps_raw) - duplicates_after_normalization
assert not bps_clean.duplicated().any()
assert validation_report.loc[
    validation_report['verificacao'] == 'ano divergente da data da compra',
    'resultado',
].iat[0] == 0
assert not bps_clean['flag_preco_total_divergente'].any()

# Atalho para as próximas etapas de análise e visualização.
bps = bps_clean

display(validation_report)
display(bps_clean.head())

,verificacao,resultado
0,registros na base bruta,342716
1,duplicatas removidas,19
2,registros na base limpa,342697
3,datas da compra inválidas,0
4,ano divergente da data da compra,0
5,inserções anteriores à compra,12
6,preços totais divergentes da fórmula,0
7,quantidades ou preços negativos,0


,ano_compra,nome_instituicao,esfera,cnpj_instituicao,municipio_instituicao,uf,compra,insercao,codigo_br,descricao_catmat,...,cnpj_fornecedor,fornecedor,cnpj_fabricante,fabricante,qtd_itens_comprados,preco_unitario,preco_total,arquivo_origem,flag_insercao_anterior_compra,flag_preco_total_divergente
0,2020,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18478187000107,MARABA,PA,2020-01-01,2024-01-19,270019,"GLICONATO DE CÁLCIO, DOSAGEM:10%, APRESENTAÇÃO...",...,04949905000163,F CARDOSO E CIA LTDA,01571702000198,HALEX ISTAR INDUSTRIA FARMACEUTICA SA,9750,4.500,43875.0,2020.csv,False,False
1,2020,FUNDO MUNICIPAL DE SAUDE - MUNICIPIO DE ALTO P...,MUNICIPAL,08533932000101,ALTO PARANA,PR,2020-01-01,2020-03-13,243488,"BOLSA VENTILAÇÃO PULMONAR, MATERIAL:BORRACHA, ...",...,06974929000106,NOROESTE MEDICAMENTOS LTDA,73856593000166,"PRATI, DONADUZZI & CIA LTDA",3,6.900,20.7,2020.csv,False,False
2,2020,MUNICIPIO DE AMERICO BRASILIENSE,MUNICIPAL,43976166000150,AMERICO BRASILIENSE,SP,2020-01-01,2020-10-06,267567,"CARVEDILOL, DOSAGEM:25 MG",...,01140868000150,CIRURGICA OLIMPIO LTDA,61150447000131,LABORATORIOS BALDACCI LTDA EM RECUPERACAO JUDI...,36000,0.155,5580.0,2020.csv,False,False
3,2020,MUNICIPIO DE AMERICO BRASILIENSE,MUNICIPAL,43976166000150,AMERICO BRASILIENSE,SP,2020-01-01,2020-10-06,271356,"ALPRAZOLAM, DOSAGEM:1 MG",...,01140868000150,CIRURGICA OLIMPIO LTDA,05254971000181,ZYDUS NIKKHO FARMACEUTICA LTDA,240000,0.099,23760.0,2020.csv,False,False
4,2020,MUNICIPIO DE AMERICO BRASILIENSE,MUNICIPAL,43976166000150,AMERICO BRASILIENSE,SP,2020-01-01,2020-10-06,267565,"CARVEDILOL, DOSAGEM:6,25 MG",...,01140868000150,CIRURGICA OLIMPIO LTDA,61150447000131,LABORATORIOS BALDACCI LTDA EM RECUPERACAO JUDI...,36000,0.086,3096.0,2020.csv,False,False


# Análise exploratória dos dados

## 6. Perguntas de negócio

1. Como o valor e a quantidade registrados evoluíram entre 2020 e 2026?
2. Quais estados, municípios e instituições concentram o maior volume financeiro?
3. Quais produtos apresentam maior valor total e maior quantidade adquirida?
4. Quais fornecedores e fabricantes possuem maior participação nos registros?
5. Quais modalidades e tipos de compra são mais utilizados?
6. Quais produtos comparáveis apresentam maior variação de preço unitário?
7. Quais registros merecem investigação adicional sem pressupor irregularidade?

## 7. Visão geral e cobertura dos dados

A cobertura temporal é conferida antes das comparações. O ano de 2026 é parcial e não deve ser comparado diretamente a anos completos sem essa ressalva.

In [9]:
measure_columns = [
    'qtd_itens_comprados',
    'preco_unitario',
    'preco_total',
]

numeric_summary = (
    bps[measure_columns]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.99])
    .round(2)
    .T
)

temporal_coverage = (
    bps.groupby('ano_compra', observed=True)
    .agg(
        primeira_compra=('compra', 'min'),
        ultima_compra=('compra', 'max'),
        registros=('compra', 'size'),
    )
    .reset_index()
)
temporal_coverage['ano_parcial'] = (
    temporal_coverage['ultima_compra'].dt.month.lt(12)
    | temporal_coverage['ultima_compra'].dt.day.lt(28)
)

cardinality_report = pd.DataFrame({
    'dimensao': [
        'UFs',
        'municípios',
        'instituições',
        'produtos CATMAT',
        'fornecedores',
        'fabricantes',
        'modalidades de compra',
    ],
    'quantidade_distinta': [
        bps['uf'].nunique(),
        bps[['uf', 'municipio_instituicao']].drop_duplicates().shape[0],
        bps['cnpj_instituicao'].nunique(),
        bps['codigo_br'].nunique(),
        bps['cnpj_fornecedor'].nunique(),
        bps['cnpj_fabricante'].nunique(),
        bps['modalidade_compra'].nunique(),
    ],
})

display(numeric_summary)
display(temporal_coverage)
display(cardinality_report)

,count,mean,std,min,25%,50%,75%,90%,99%,max
qtd_itens_comprados,342697.0,166698.70,8308189.14,1.0,250.00,1600.00,12000.00,72000.0,1800000.00,4.356140e+09
preco_unitario,342697.0,171.30,17535.11,0.0,0.35,1.85,7.17,28.5,850.00,8.191720e+06
preco_total,342697.0,229233.05,39848521.46,0.0,700.00,2800.00,13000.00,60000.0,1508666.73,2.281600e+10


,ano_compra,primeira_compra,ultima_compra,registros,ano_parcial
0,2020,2020-01-01,2020-12-31,84819,False
1,2021,2021-01-01,2021-12-31,83622,False
2,2022,2022-01-01,2022-12-30,88991,False
3,2023,2023-01-01,2023-12-30,31992,False
4,2024,2024-01-01,2024-12-30,26242,False
5,2025,2025-01-01,2025-12-30,26214,False
6,2026,2026-01-06,2026-03-05,817,True


,dimensao,quantidade_distinta
0,UFs,24
1,municípios,737
2,instituições,831
3,produtos CATMAT,12994
4,fornecedores,3502
5,fabricantes,2290
6,modalidades de compra,9


## 8. KPIs obrigatórios

O preço unitário médio ponderado é calculado como valor total dividido pela quantidade total. Ele só deve ser interpretado dentro de filtros que mantenham produtos, apresentações e unidades comparáveis.

In [16]:
total_value = bps['preco_total'].sum().round(2)
total_quantity = bps['qtd_itens_comprados'].sum()
weighted_unit_price = total_value / total_quantity if total_quantity else pd.NA

kpis = pd.DataFrame([
    {
        'kpi': 'Valor total registrado',
        'valor': total_value,
        'unidade': 'R$',
        'definicao': 'Soma de preco_total',
    },
    {
        'kpi': 'Quantidade total de itens comprados',
        'valor': total_quantity,
        'unidade': 'itens',
        'definicao': 'Soma de qtd_itens_comprados',
    },
    {
        'kpi': 'Número de registros de compra',
        'valor': len(bps),
        'unidade': 'registros',
        'definicao': 'Contagem de linhas da base limpa',
    },
    {
        'kpi': 'Instituições compradoras',
        'valor': bps['cnpj_instituicao'].nunique(),
        'unidade': 'instituições distintas',
        'definicao': 'Contagem distinta de CNPJ da instituição',
    },
    {
        'kpi': 'Fornecedores',
        'valor': bps['cnpj_fornecedor'].nunique(),
        'unidade': 'fornecedores distintos',
        'definicao': 'Contagem distinta de CNPJ do fornecedor',
    },
    {
        'kpi': 'Preço unitário médio ponderado',
        'valor': weighted_unit_price,
        'unidade': 'R$/item',
        'definicao': 'Valor total dividido pela quantidade total',
    },
])

recomputed_total = (bps['qtd_itens_comprados'] * bps['preco_unitario']).sum()
reconciliation_tolerance = 0.01 + abs(total_value) * 1e-9
assert abs(total_value - recomputed_total) <= reconciliation_tolerance
assert int(kpis.loc[kpis['kpi'] == 'Número de registros de compra', 'valor'].iat[0]) == len(bps)

display(kpis)

,kpi,valor,unidade,definicao
0,Valor total registrado,7.855748e+10,R$,Soma de preco_total
1,Quantidade total de itens comprados,5.712714e+10,itens,Soma de qtd_itens_comprados
2,Número de registros de compra,3.426970e+05,registros,Contagem de linhas da base limpa
3,Instituições compradoras,8.310000e+02,instituições distintas,Contagem distinta de CNPJ da instituição
4,Fornecedores,3.502000e+03,fornecedores distintos,Contagem distinta de CNPJ do fornecedor
5,Preço unitário médio ponderado,1.375134e+00,R$/item,Valor total dividido pela quantidade total


## 9. Evolução anual

In [17]:
annual_summary = (
    bps.groupby('ano_compra', observed=True)
    .agg(
        valor_total=('preco_total', 'sum'),
        quantidade_total=('qtd_itens_comprados', 'sum'),
        registros=('preco_total', 'size'),
        instituicoes=('cnpj_instituicao', 'nunique'),
        fornecedores=('cnpj_fornecedor', 'nunique'),
    )
    .reset_index()
)
annual_summary['preco_unitario_medio_ponderado'] = (
    annual_summary['valor_total'] / annual_summary['quantidade_total']
)
annual_summary = annual_summary.merge(
    temporal_coverage[['ano_compra', 'ultima_compra', 'ano_parcial']],
    on='ano_compra',
    how='left',
)

assert annual_summary['valor_total'].sum().round(2) == total_value
assert annual_summary['quantidade_total'].sum() == total_quantity
assert annual_summary['registros'].sum() == len(bps)

display(annual_summary)

,ano_compra,valor_total,quantidade_total,registros,instituicoes,fornecedores,preco_unitario_medio_ponderado,ultima_compra,ano_parcial
0,2020,5.046764e+09,8769353249,84819,482,1543,0.575500,2020-12-31,False
1,2021,7.411106e+09,10689715426,83622,475,1623,0.693293,2021-12-31,False
2,2022,1.686436e+10,15186371537,88991,416,1760,1.110493,2022-12-30,False
3,2023,4.920072e+09,10179744296,31992,272,1102,0.483320,2023-12-30,False
4,2024,8.964915e+09,5321453856,26242,230,1068,1.684674,2024-12-30,False
5,2025,3.493090e+10,6698301159,26214,174,1083,5.214889,2025-12-30,False
6,2026,4.193677e+08,282204198,817,17,137,1.486043,2026-03-05,True


## 10. Distribuição geográfica e instituições

Os rankings utilizam `preco_total` como volume financeiro registrado. Instituições são identificadas pelo CNPJ para evitar dupla contagem por variações no nome.

In [18]:
def summarize_by_value(group_columns, top_n=10):
    return (
        bps.groupby(group_columns, dropna=False, observed=True)
        .agg(
            valor_total=('preco_total', 'sum'),
            quantidade_total=('qtd_itens_comprados', 'sum'),
            registros=('preco_total', 'size'),
        )
        .reset_index()
        .nlargest(top_n, 'valor_total')
    )

top_states = summarize_by_value(['uf'], top_n=27)
top_municipalities = summarize_by_value(
    ['uf', 'municipio_instituicao'], top_n=10
)
top_institutions = summarize_by_value(
    ['cnpj_instituicao', 'nome_instituicao'], top_n=10
)

display(top_states)
display(top_municipalities)
display(top_institutions)

,uf,valor_total,quantidade_total,registros
14,PR,2.917794e+10,9491961008,76298
22,SP,2.537383e+10,19481074088,62982
3,CE,5.396203e+09,739151022,5066
15,RJ,5.163017e+09,462252407,7848
20,SC,1.939026e+09,6150398900,11286
19,RS,1.833604e+09,9995202088,17830
1,AL,1.522319e+09,2098221496,10889
4,ES,1.361654e+09,3649166797,9790
2,BA,1.254050e+09,407604424,2310
11,PB,8.572643e+08,250908769,23036


,uf,municipio_instituicao,valor_total,quantidade_total,registros
359,PR,CURITIBA,2.764886e+10,7410897477,5766
720,SP,SAO PAULO,2.204603e+10,15309065236,4804
42,CE,FORTALEZA,5.349154e+09,726449936,209
479,RJ,RIO DE JANEIRO,4.901364e+09,182999181,918
597,SC,FLORIANOPOLIS,1.802862e+09,5368369644,6702
23,AL,PENEDO,1.445117e+09,1885421353,2057
37,BA,SALVADOR,1.197435e+09,328456877,928
639,SP,BARUERI,1.115245e+09,403102155,1019
79,ES,VITORIA,1.046557e+09,2741670795,2083
577,RS,PORTO ALEGRE,1.008180e+09,8222630033,881


,cnpj_instituicao,nome_instituicao,valor_total,quantidade_total,registros
783,76416866000140,SECRETARIA DE ESTADO DA SAUDE,2.488315e+10,724175883,1562
665,46374500026231,SECRETARIA DE ESTADO DA SAUDE,2.187677e+10,14754292555,4641
128,07954571000104,SECRETARIA DA SAUDE DO ESTADO DO CEARA,5.347701e+09,726449418,206
607,42498717000155,SECRETARIA DE ESTADO DE SAUDE - SES,4.901239e+09,182975824,915
142,08597121000174,FUNDO ESTADUAL DE SAUDE DO PARANA - FUNSAUDE,1.698618e+09,430130404,1704
443,12075748000132,CONSORCIO INTERMUNICIPAL CATARINENSE-CIMCATARINA,1.481930e+09,4783679034,5964
555,18538208000124,CONSORCIO INTERMUNICIPAL DO SUL DO ESTADO DE A...,1.445117e+09,1885421353,2057
100,05816630000152,FUNDO ESTADUAL DE SAUDE DO ESTADO DA BAHIA,1.175198e+09,328417958,916
678,46523015000135,MUNICIPIO DE BARUERI,1.114840e+09,401818155,1013
590,27080605000196,SECRETARIA DE ESTADO DA SAUDE,1.015112e+09,2596256255,1748


## 11. Produtos, fornecedores, fabricantes e modalidades

In [19]:
product_dimensions = [
    'codigo_br',
    'descricao_catmat',
    'unidade_fornecimento_capacidade',
]

product_summary = (
    bps.groupby(product_dimensions, dropna=False, observed=True)
    .agg(
        valor_total=('preco_total', 'sum'),
        quantidade_total=('qtd_itens_comprados', 'sum'),
        registros=('preco_total', 'size'),
        fornecedores=('cnpj_fornecedor', 'nunique'),
    )
    .reset_index()
)
top_products_by_value = product_summary.nlargest(10, 'valor_total')
top_products_by_quantity = product_summary.nlargest(10, 'quantidade_total')

top_suppliers = summarize_by_value(
    ['cnpj_fornecedor', 'fornecedor'], top_n=10
)
top_manufacturers = summarize_by_value(
    ['cnpj_fabricante', 'fabricante'], top_n=10
)
purchase_modes = summarize_by_value(['modalidade_compra'], top_n=20)
purchase_types = summarize_by_value(['tipo_compra'], top_n=20)

display(top_products_by_value)
display(top_products_by_quantity)
display(top_suppliers)
display(top_manufacturers)
display(purchase_modes)
display(purchase_types)

,codigo_br,descricao_catmat,unidade_fornecimento_capacidade,valor_total,quantidade_total,registros,fornecedores
1452,272815,"PENICILAMINA, DOSAGEM:250 MG",CÁPSULA,2.284796e+10,1292690,15,5
3881,356051,"ÁCIDO ZOLEDRÔNICO, CONCENTRAÇÃO:50 MCG/ML, FOR...",BOLSA 100.00 ML,3.719442e+09,53875,8,6
14461,606824,"TEZEPELUMABE, CONCENTRAÇÃO:110 MG/ML, FORMA FA...",SERINGA 1.91 ML,1.665818e+09,571,10,2
7938,429724,"CARFILZOMIB, CONCENTRAÇÃO:60 MG, FORMA FARMACE...",FRASCO-AMPOLA,1.380291e+09,4645,13,4
8274,432908,"DAPAGLIFLOZINA, CONCENTRAÇÃO:10 MG",COMPRIMIDO,1.047319e+09,487353412,232,83
5661,403131,"BUDESONIDA, CONCENTRAÇÃO:50 MCG/DOSE, FORMA FA...",FRASCO 200.00 DOSES,9.765972e+08,79246121,47,19
9525,439252,"NUSINERSENA, CONCENTRAÇÃO:2,4 MG/ML, FORMA FAR...",FRASCO 5.00 ML,9.305701e+08,3089,12,5
9526,439259,"AXITINIBE, CONCENTRAÇÃO:5 MG",COMPRIMIDO,7.212140e+08,30280,18,13
8825,436345,"NINTEDANIBE, COMPOSIÇÃO:SAL ESILATO, CONCENTRA...",CÁPSULA,6.719030e+08,3052950,80,33
863,269979,"COMPRESSA GAZE, MATERIAL:TECIDO 100% ALGODÃO, ...",PACOTE 5.00 UN,6.227361e+08,6401000,4,4


,codigo_br,descricao_catmat,unidade_fornecimento_capacidade,valor_total,quantidade_total,registros,fornecedores
5986,404992,"DIETA ENTERAL, ASPECTO FÍSICO:LÍQUIDO, USO:ENT...",MILILITRO,5.459607e+07,4357306650,10,9
652,268856,"LOSARTANA POTÁSSICA, DOSAGEM:50 MG",COMPRIMIDO,1.275422e+08,1993391651,897,281
339,267674,"HIDROCLOROTIAZIDA, DOSAGEM:25 MG",COMPRIMIDO,4.339457e+07,1669197466,805,240
8575,435227,"DIETA ENTERAL, ASPECTO FÍSICO:LÍQUIDO, USO:ENT...",MILILITRO,3.575959e+07,1438409000,10,7
244,267512,"AMITRIPTILINA CLORIDRATO, DOSAGEM:25 MG",COMPRIMIDO,7.382055e+07,1298155875,974,236
354,267712,"OMEPRAZOL, CONCENTRAÇÃO:20 MG",CÁPSULA,8.854693e+07,1155381113,739,226
1384,272434,"ANLODIPINO BESILATO, DOSAGEM:5 MG",COMPRIMIDO,3.201836e+07,1144276427,915,243
233,267502,"ÁCIDO ACETILSALICÍLICO, DOSAGEM:100 MG",COMPRIMIDO,4.282577e+07,1142302596,986,278
348,267691,"METFORMINA CLORIDRATO, DOSAGEM:850 MG",COMPRIMIDO,9.525090e+07,1119671715,800,239
185,267203,"DIPIRONA SÓDICA, DOSAGEM:500 MG",COMPRIMIDO,1.229824e+08,1085070801,972,264


,cnpj_fornecedor,fornecedor,valor_total,quantidade_total,registros
1155,11697594000310,AGILLE COMERCIO DE MEDICAMENTOS LTDA,2.284569e+10,1295182,19
2078,27817504000155,SP HOSPITALAR LTDA,3.829422e+09,169024801,170
3260,60318797000100,ASTRAZENECA DO BRASIL LTDA.,3.439432e+09,460018832,200
3121,51780468000268,JANSSEN-CILAG FARMACEUTICA LTDA,1.881164e+09,9130752,250
3212,56994502002779,NOVARTIS BIOCIENCIAS SA,1.767763e+09,70622926,235
1014,10586940000168,ONCOVIT DISTRIBUIDORA DE MEDICAMENTOS LTDA,1.674614e+09,96364369,511
2936,44734671000151,CRISTALIA PRODUTOS QUIMICOS FARMACEUTICOS LTDA,1.029646e+09,1376274266,6233
376,04274988000138,ATIVA COMERCIAL HOSPITALAR LTDA,1.029269e+09,123946690,420
886,09053134000145,ELFA MEDICAMENTOS S.A,9.907257e+08,70343834,324
385,04307650001298,ONCO PROD DISTRIBUIDORA DE PRODUTOS HOSPITALAR...,9.336751e+08,75765187,385


,cnpj_fabricante,fabricante,valor_total,quantidade_total,registros
1159,17875154000120,MEDQUIMICA INDUSTRIA FARMACEUTICA LTDA.,2.290336e+10,794527742,2005
2013,61190096000192,EUROFARMA LABORATORIOS S.A.,5.684232e+09,654632414,6348
1899,56994502000130,NOVARTIS BIOCIENCIAS SA,3.580460e+09,204866326,3528
1960,60318797000100,ASTRAZENECA DO BRASIL LTDA.,3.568440e+09,588240253,1562
1839,51780468000187,JANSSEN-CILAG FARMACEUTICA LTDA,2.195234e+09,9067123,778
1192,18774815000193,AMGEN BIOTECNOLOGIA DO BRASIL LTDA.,1.712797e+09,733604,216
1999,61072393000133,PFIZER BRASIL LTDA,1.463418e+09,8857868,552
877,10588595001092,SANOFI MEDLEY FARMACEUTICA LTDA,1.402905e+09,352222839,2844
2156,73856593000166,"PRATI, DONADUZZI & CIA LTDA",1.357228e+09,8066383611,22646
1733,44734671000151,CRISTALIA PRODUTOS QUIMICOS FARMACEUTICOS LTDA,1.284724e+09,1955569996,15514


,modalidade_compra,valor_total,quantidade_total,registros
6,Pregão,7.378499e+10,53569018547,315340
7,Registro de Preços,2.719405e+09,1780018823,10938
3,Dispensa de Licitação,1.842824e+09,1728465947,13174
4,Inexigibilidade de Licitação,1.885204e+08,13667578,416
8,Tomada de Preços,1.014499e+07,14262632,1784
0,Concorrência,6.030425e+06,7122609,722
1,Concurso,2.659454e+06,5280070,196
5,Leilão,2.149697e+06,9202674,57
2,Convite,7.496313e+05,104841,70


,tipo_compra,valor_total,quantidade_total,registros
0,ADMINISTRATIVA,7.178229e+10,56630293749,337485
1,JUDICIAL,6.775189e+09,496849972,5212


## 12. Variação de preços entre produtos comparáveis

A comparação agrupa o mesmo código CATMAT e a mesma unidade/capacidade. São considerados apenas grupos com pelo menos 10 registros, dois fornecedores e preços positivos. A classificação utiliza a razão entre os percentis 90 e 10, que é menos sensível a valores extremos do que máximo/mínimo. Uma razão elevada indica somente uma oportunidade de investigação; não comprova sobrepreço, economia ou irregularidade.

In [20]:
price_variation = (
    bps.groupby(product_dimensions, dropna=False, observed=True)
    .agg(
        registros=('preco_unitario', 'size'),
        fornecedores=('cnpj_fornecedor', 'nunique'),
        quantidade_total=('qtd_itens_comprados', 'sum'),
        valor_total=('preco_total', 'sum'),
        preco_minimo=('preco_unitario', 'min'),
        preco_p10=('preco_unitario', lambda values: values.quantile(0.10)),
        preco_mediano=('preco_unitario', 'median'),
        preco_p90=('preco_unitario', lambda values: values.quantile(0.90)),
        preco_maximo=('preco_unitario', 'max'),
    )
    .reset_index()
)
price_variation['preco_medio_ponderado'] = (
    price_variation['valor_total'] / price_variation['quantidade_total']
)
price_variation['razao_max_min'] = (
    price_variation['preco_maximo'] / price_variation['preco_minimo']
)
price_variation['razao_p90_p10'] = (
    price_variation['preco_p90'] / price_variation['preco_p10']
)

price_investigation_candidates = (
    price_variation.loc[
        price_variation['registros'].ge(10)
        & price_variation['fornecedores'].ge(2)
        & price_variation['preco_minimo'].gt(0)
    ]
    .nlargest(20, 'razao_p90_p10')
)

assert price_investigation_candidates['registros'].ge(10).all()
assert price_investigation_candidates['fornecedores'].ge(2).all()
assert price_investigation_candidates['preco_minimo'].gt(0).all()
assert price_investigation_candidates['preco_p10'].gt(0).all()

display(price_investigation_candidates)

,codigo_br,descricao_catmat,unidade_fornecimento_capacidade,registros,fornecedores,quantidade_total,valor_total,preco_minimo,preco_p10,preco_mediano,preco_p90,preco_maximo,preco_medio_ponderado,razao_max_min,razao_p90_p10
8169,432316,"DIETA INFANTIL, INDICAÇÃO:0 A 12 MESES, ASPECT...",GRAMA,17,12,4928841,2214436.04,0.0350,0.04140,22.460,287.5600,1000.000,0.449281,28571.428571,6945.893720
12099,461250,"ELETRODO USO MÉDICO, APLICAÇÃO 1:P/ MONITORIZA...",UNIDADE,11,11,58133,88450.05,0.3000,0.30000,0.390,1750.0000,1900.000,1.521512,6333.333333,5833.333333
5971,404934,"DIETA ENTERAL, ASPECTO FÍSICO:PÓ, USO:ENTERAL ...",GRAMA,24,16,4541921,1436716.90,0.0417,0.05920,66.000,325.0000,329.000,0.316324,7889.688249,5489.864865
5973,404944,"DIETA ENTERAL, ASPECTO FÍSICO:PÓ, USO:ENTERAL ...",GRAMA,30,18,85612352,5118215.20,0.0320,0.05675,42.150,171.5400,321.940,0.059784,10060.625000,3022.731278
12389,465751,"DIETA INFANTIL, INDICAÇÃO:A PARTIR 3 ANOS, ASP...",GRAMA,13,9,1753798,341412.01,0.0565,0.08802,39.870,215.7000,348.470,0.194670,6167.610619,2450.579414
5986,404992,"DIETA ENTERAL, ASPECTO FÍSICO:LÍQUIDO, USO:ENT...",MILILITRO,10,9,4357306650,54596071.00,0.0125,0.01592,12.480,30.4679,83.999,0.012530,6719.920000,1913.812814
5821,403932,"DIETA ENTERAL, INDICAÇÃO:MÓDULO DE CHO, ASPECT...",GRAMA,14,14,14128622,549443.80,0.0300,0.03333,11.800,62.8250,157.980,0.038889,5266.000000,1884.938494
12280,464185,"DIETA INFANTIL, INDICAÇÃO:1 A 10 ANOS, ASPECTO...",GRAMA,11,11,3423966,6484420.48,0.0579,0.06000,40.000,110.0000,188.720,1.893833,3259.412781,1833.333333
8822,436335,"DIETA INFANTIL, INDICAÇÃO:ACIMA DE 1 ANO, ASPE...",GRAMA,12,8,1063575,872048.20,0.0491,0.07900,0.516,140.9720,215.000,0.819922,4378.818737,1784.455696
8717,435949,"DIETA INFANTIL, INDICAÇÃO:0 A 12 MESES, ASPECT...",GRAMA,10,9,617738,609695.42,0.0347,0.05117,10.045,90.2550,93.000,0.986981,2680.115274,1763.826461


## 13. Limitações preliminares

- O BPS contém valores registrados pelas instituições e não deve ser interpretado automaticamente como o universo completo das compras em saúde.
- O ano de 2026 possui cobertura parcial até a última data observada na base.
- O preço unitário médio ponderado global mistura produtos e apresentações distintas; seu uso analítico exige filtros de comparabilidade.
- Valores extremos devem ser investigados juntamente com fabricante, fornecedor, modalidade, quantidade, localidade e período.
- Diferenças de preço não constituem, isoladamente, evidência de economia, sobrepreço ou irregularidade.
- Campos como `capacidade`, `unidade_medida`, `generico` e `anvisa` apresentam ausência relevante e limitam algumas comparações.

## 14. Exportação da base para o dashboard

A base tratada é exportada em CSV UTF-8, separado por vírgulas e com datas no padrão `YYYY-MM-DD`. O arquivo é gerado em `output/data/` e permanece fora do Git devido ao tamanho. O esquema versionado em `config/bigquery_schema.json` deve ser usado na carga para o BigQuery.

In [21]:
import csv
import hashlib
import json

output_dir = project_dir / 'output' / 'data'
output_dir.mkdir(parents=True, exist_ok=True)
dashboard_file = output_dir / 'BPS_20_26_SamuelBucco.csv'
schema_file = project_dir / 'config' / 'bigquery_schema.json'

# O upload de CSV do Looker Studio não aceita quebras de linha internas.
dashboard_text_columns = bps.select_dtypes(include=['object', 'string']).columns
internal_line_breaks = {
    column: int(
        bps[column]
        .astype('string')
        .str.contains(r'[\r\n]', regex=True, na=False)
        .sum()
    )
    for column in dashboard_text_columns
}
internal_line_breaks = {
    column: count
    for column, count in internal_line_breaks.items()
    if count > 0
}
if internal_line_breaks:
    raise ValueError(
        f'Quebras de linha internas encontradas: {internal_line_breaks}'
    )

bps.to_csv(
    dashboard_file,
    index=False,
    encoding='utf-8',
    sep=',',
    date_format='%Y-%m-%d',
    na_rep='',
)

with schema_file.open(encoding='utf-8') as stream:
    bigquery_schema = json.load(stream)
schema_columns = [field['name'] for field in bigquery_schema]

with dashboard_file.open(encoding='utf-8', newline='') as stream:
    reader = csv.reader(stream)
    exported_header = next(reader)
    exported_rows = sum(1 for _ in reader)

sha256 = hashlib.sha256()
with dashboard_file.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        sha256.update(chunk)

file_size_mb = dashboard_file.stat().st_size / 1024**2
export_validation = pd.DataFrame([
    {'verificacao': 'arquivo gerado', 'resultado': str(dashboard_file)},
    {'verificacao': 'linhas exportadas', 'resultado': exported_rows},
    {'verificacao': 'colunas exportadas', 'resultado': len(exported_header)},
    {'verificacao': 'tamanho em MB', 'resultado': round(file_size_mb, 2)},
    {'verificacao': 'SHA-256', 'resultado': sha256.hexdigest()},
    {
        'verificacao': 'compatível com limite de 100 MB do upload direto',
        'resultado': file_size_mb <= 100,
    },
])

assert exported_rows == len(bps)
assert exported_header == bps.columns.tolist()
assert schema_columns == exported_header
assert len(exported_header) == bps.shape[1]

display(export_validation)

,verificacao,resultado
0,arquivo gerado,/Users/samuelbucco/Documents/SCTec/Análise de...
1,linhas exportadas,342697
2,colunas exportadas,28
3,tamanho em MB,134.48
4,SHA-256,278233ec087aada9f1fa47a2aa071fc866ba09f0c21271...
5,compatível com limite de 100 MB do upload direto,False


## 15. Exportação enxuta para Google Sheets

Para o projeto acadêmico, uma segunda fonte preserva somente os campos usados pelos KPIs, filtros e visuais do dashboard. A redução evita operar próximo ao limite de 10 milhões de células do Google Sheets sem substituir a base completa de auditoria.

In [22]:
import numpy as np

sheets_source_columns = [
    'ano_compra',
    'compra',
    'uf',
    'municipio_instituicao',
    'cnpj_instituicao',
    'nome_instituicao',
    'esfera',
    'codigo_br',
    'descricao_catmat',
    'unidade_fornecimento_capacidade',
    'modalidade_compra',
    'tipo_compra',
    'cnpj_fornecedor',
    'fornecedor',
    'cnpj_fabricante',
    'fabricante',
    'qtd_itens_comprados',
    'preco_unitario',
    'preco_total',
]

sheets_data = (
    bps.loc[:, sheets_source_columns]
    .rename(columns={
        'cnpj_instituicao': 'id_instituicao',
        'cnpj_fornecedor': 'id_fornecedor',
        'cnpj_fabricante': 'id_fabricante',
    })
    .copy()
)
sheets_data['id_instituicao'] = 'CNPJ ' + sheets_data['id_instituicao'].astype('string')
sheets_data['id_fornecedor'] = 'CNPJ ' + sheets_data['id_fornecedor'].astype('string')
sheets_data['id_fabricante'] = 'CNPJ ' + sheets_data['id_fabricante'].astype('string')
sheets_data['codigo_br'] = 'BR ' + sheets_data['codigo_br'].astype('string')
sheets_columns = [
    'ano_compra',
    'compra',
    'uf',
    'municipio_instituicao',
    'id_instituicao',
    'nome_instituicao',
    'esfera',
    'codigo_br',
    'descricao_catmat',
    'unidade_fornecimento_capacidade',
    'modalidade_compra',
    'tipo_compra',
    'id_fornecedor',
    'fornecedor',
    'id_fabricante',
    'fabricante',
    'qtd_itens_comprados',
    'preco_unitario',
    'preco_total',
]
sheets_data = sheets_data.loc[:, sheets_columns]
sheets_file = output_dir / 'BPS_20_26_SamuelBucco_GoogleSheets.csv'
sheets_data.to_csv(
    sheets_file,
    index=False,
    encoding='utf-8',
    sep=',',
    date_format='%Y-%m-%d',
    na_rep='',
)

sheets_identifier_columns = [
    'id_instituicao',
    'codigo_br',
    'id_fornecedor',
    'id_fabricante',
]
sheets_loaded = pd.read_csv(
    sheets_file,
    dtype={column: 'string' for column in sheets_identifier_columns},
    parse_dates=['compra'],
)

sheets_limit_cells = 10_000_000
sheets_total_cells = (len(sheets_loaded) + 1) * len(sheets_loaded.columns)
sheets_occupancy = sheets_total_cells / sheets_limit_cells
sheets_weighted_price = (
    sheets_loaded['preco_total'].sum()
    / sheets_loaded['qtd_itens_comprados'].sum()
)
source_weighted_price = bps['preco_total'].sum() / bps['qtd_itens_comprados'].sum()

sheets_sha256 = hashlib.sha256()
with sheets_file.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        sheets_sha256.update(chunk)

assert sheets_loaded.columns.tolist() == sheets_columns
assert len(sheets_loaded) == len(bps)
assert sheets_total_cells < sheets_limit_cells
assert np.isclose(sheets_loaded['preco_total'].sum(), bps['preco_total'].sum())
assert sheets_loaded['qtd_itens_comprados'].sum() == bps['qtd_itens_comprados'].sum()
assert sheets_loaded['id_instituicao'].nunique() == bps['cnpj_instituicao'].nunique()
assert sheets_loaded['id_fornecedor'].nunique() == bps['cnpj_fornecedor'].nunique()
assert sheets_loaded['id_fabricante'].nunique() == bps['cnpj_fabricante'].nunique()
assert sheets_loaded['id_instituicao'].str.startswith('CNPJ ', na=False).all()
assert sheets_loaded['id_fornecedor'].str.startswith('CNPJ ', na=False).all()
assert sheets_loaded['id_fabricante'].str.startswith('CNPJ ', na=False).all()
assert sheets_loaded['codigo_br'].str.startswith('BR ', na=False).all()
assert np.isclose(sheets_weighted_price, source_weighted_price)

sheets_export_validation = pd.DataFrame([
    {'verificacao': 'arquivo gerado', 'resultado': str(sheets_file)},
    {'verificacao': 'linhas exportadas', 'resultado': len(sheets_loaded)},
    {'verificacao': 'colunas exportadas', 'resultado': len(sheets_loaded.columns)},
    {'verificacao': 'células com cabeçalho', 'resultado': sheets_total_cells},
    {'verificacao': 'ocupação do limite', 'resultado': f'{sheets_occupancy:.2%}'},
    {'verificacao': 'tamanho em MB', 'resultado': round(sheets_file.stat().st_size / 1024**2, 2)},
    {'verificacao': 'SHA-256', 'resultado': sheets_sha256.hexdigest()},
])

display(sheets_export_validation)

,verificacao,resultado
0,arquivo gerado,/Users/samuelbucco/Documents/SCTec/Análise de...
1,linhas exportadas,342697
2,colunas exportadas,19
3,células com cabeçalho,6511262
4,ocupação do limite,65.11%
5,tamanho em MB,120.41
6,SHA-256,ccd5077f7f218785a34aa7ce34d3c4d5498264b4c7bc48...
